In [1]:
# pip install openai
from openai import OpenAI
import os

client = OpenAI(
    # 若没有配置环境变量，请用百炼API Key将下行替换为：api_key="sk-xxx"
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    # 模型列表：https://help.aliyun.com/zh/model-studio/getting-started/models
    model="deepseek-v4-flash",
    messages=[
        {"role": "system", "content": "你是一个人工智能教授，会写代码，能够讲清楚理论"},
        {"role": "user", "content": "请解释Transformer的架构原理"},
    ],
)
print(completion.choices[0].message.content)

作为人工智能教授，我将从原理到代码实现为你系统讲解Transformer的架构。Transformer由Vaswani等人在2017年提出，核心思想是完全基于注意力机制，抛弃了循环和卷积，实现并行化序列建模。下面分为几个部分展开。

---

## 1. 为什么需要Transformer？

传统RNN/LSTM处理序列时按时间步逐步计算，无法并行，长距离依赖易遗忘。CNN虽可并行但感受野受限。Transformer通过**自注意力（Self-Attention）**解决这些问题，每个位置可以直接与所有位置交互，计算复杂度为O(n²)但可完全并行。

---

## 2. 总体架构（Encoder-Decoder）

Transformer由堆叠的编码器和解码器组成。  
- **编码器**：6个相同层，每层包含：多头自注意力 + 前馈神经网络，每个子层后接残差连接和层归一化（Add & Norm）。  
- **解码器**：6个相同层，每层包含：掩码多头自注意力 + 交叉多头注意力（key, value来自编码器输出） + 前馈神经网络，每个子层后同样有Add & Norm。

---

## 3. 核心模块详解

### 3.1 缩放点积注意力（Scaled Dot-Product Attention）

公式：
```
Attention(Q, K, V) = softmax( Q K^T / √d_k ) V
```
其中 Q, K, V 由输入经过线性变换得到。除以 √d_k 防止内积过大导致softmax梯度消失。

### 3.2 多头注意力（Multi-Head Attention）

将Q, K, V投影到h个不同的子空间（通常h=8），分别做注意力，然后拼接并线性变换：
```
MultiHead(Q,K,V) = Concat(head_1,...,head_h) W_O
head_i = Attention(Q W_Q^i, K W_K^i, V W_V^i)
```
每个头关注不同的模式，增强表示能力。

### 3.3 位置编码（Positional Encoding）

由于注意力本身没有顺序信息，需要显式注入位置。采用正弦和余弦函数：
```
PE(pos, 2i) = sin(pos / 10000^{2i/d_model})
PE